# Lean-15b : Grothendieck en Lean -- Atelier pratique

**Navigation** : [<< Lean-15 Grothendieck Tribute](Lean-15-Grothendieck-Tribute.ipynb) | [Lean-16b Conway Tribute >>](Lean-16b-Conway-Game-of-Life-Lean.ipynb) | [Index](README.md)

**Kernel** : Python 3 (sources Lean lues depuis `grothendieck_lean/` + execution de snippets via subprocess WSL)

---

> *On peut tout faire pourvu qu'on prenne le temps de comprendre les choses.* -- A. Grothendieck

## Objectifs d'apprentissage

Ce notebook est le **complement pratique** du [Lean-15 (Hommage)](Lean-15-Grothendieck-Tribute.ipynb). La ou Lean-15 presente le contexte biographique et mathematique, Lean-15b vous fait **manipuler directement** les modules Lean du projet `grothendieck_lean/`.

A la fin de ce notebook, vous saurez :

1. Naviguer dans le projet Lake `grothendieck_lean/` et comprendre sa structure de modules.
2. Lire les sources Lean des 6 modules pedagogiques (parmi les 23 du projet) et identifier les constructions cles (cribles, topologies, schemas, site de Zariski).
3. Analyser les 4 micro-preuves de Calibration (P1-P4) et comprendre les tactiques utilisees.
4. Interpreter les identites de pullback dans le treillis des cribles.
5. Utiliser la carte MathlibMap comme index de reference des structures grothendieckiennes disponibles.
6. Explorer interactivement les definitions via des snippets Lean executes en WSL.

### Prerequis

- Avoir lu le [Lean-15 (Hommage Grothendieck)](Lean-15-Grothendieck-Tribute.ipynb) pour le contexte mathematique.
- Familiarite avec Lean 4 et Mathlib (cf Lean-1 a Lean-6).
- Le projet Lake `grothendieck_lean/` doit etre present dans le meme repertoire que ce notebook.

### Duree estimee : 45 minutes

**Note technique**

Ce notebook utilise un kernel **Python 3**. Les sources Lean sont lues directement depuis le projet `grothendieck_lean/` (acces fichier Windows). Les snippets interactifs sont executes via `subprocess` -> WSL -> `lake env lean`. Ce pattern est emprunte aux notebooks Lean-13/15/16 de la serie.

**Important** : les cellules `run_lean()` executent du code dans l'environnement Lake du projet. Le premier appel peut etre long (chargement Mathlib). Les cellules `display_lean_module()` fonctionnent immediatement (lecture fichier uniquement).

In [1]:
import subprocess
import tempfile
import textwrap
import re
import os
from pathlib import Path

# --- Path resolution: find the grothendieck_lean Lake project ---
# Follows the same pattern as Lean-13/15/16 (Kochen-Specker/Grothendieck/Conway).
# Must work both interactively (CWD = repo root) and under Papermill (CWD may differ).

def find_grothendieck_lean_project():
    """Find the grothendieck_lean Lake project directory.

    Searches from multiple starting points to handle both interactive use
    and Papermill execution (where CWD may differ from notebook location).
    Returns an ABSOLUTE path.
    """
    starts = [Path.cwd().resolve()]

    nb_file = os.environ.get('NB_FILE') or globals().get('__vsc_ipynb_file__')
    if nb_file:
        starts.append(Path(nb_file).resolve().parent)

    for start in starts:
        current = start
        for _ in range(10):
            candidate = current / 'grothendieck_lean'
            if candidate.exists() and (candidate / 'lakefile.lean').exists():
                return candidate.resolve()
            current = current.parent
            if current == current.parent:
                break
    raise FileNotFoundError("grothendieck_lean/ not found -- check working directory")

def win_to_wsl(win_path: Path) -> str:
    """Convert Windows path to WSL path using drive letter."""
    p = win_path.resolve()
    drive_letter = p.drive
    if not drive_letter or len(drive_letter) < 2:
        s = str(p)
        if s.startswith('/mnt/'):
            return s
        drive_letter = 'D:'
    drive = drive_letter[0].lower()
    return f'/mnt/{drive}{p.as_posix()[2:]}'

WIN_LEAN_PROJECT = find_grothendieck_lean_project()
LEAN_PROJECT = win_to_wsl(WIN_LEAN_PROJECT)

def wsl(cmd, timeout=60):
    """Run a bash command inside WSL Ubuntu.

    Captures stdout/stderr via temp files rather than capture_output=True, to
    avoid the CPython ``_readerthread`` race on Windows that silently dropped
    subprocess output (cells 8/12/16/28/30/32 previously showed only an
    ``Exception in thread (_readerthread)`` trace instead of Lean output).
    Same fix as Lean-15-Grothendieck-Tribute PR #3216.
    """
    import tempfile
    full = ['wsl', '-d', 'Ubuntu', '--', 'bash', '-lc', cmd]
    out_f = tempfile.NamedTemporaryFile('wb', delete=False, suffix='.out')
    err_f = tempfile.NamedTemporaryFile('wb', delete=False, suffix='.err')
    out_path, err_path = out_f.name, err_f.name
    out_f.close()
    err_f.close()
    try:
        with open(out_path, 'wb') as o, open(err_path, 'wb') as e:
            r = subprocess.run(full, stdout=o, stderr=e, timeout=timeout)
        out = Path(out_path).read_text(encoding='utf-8', errors='replace')
        err = Path(err_path).read_text(encoding='utf-8', errors='replace')
        return r.returncode, out, err
    except FileNotFoundError:
        return 127, '', 'WSL executable not found'
    except subprocess.TimeoutExpired:
        return -1, '', f'TIMEOUT after {timeout}s'
    finally:
        for p in (out_path, err_path):
            try:
                Path(p).unlink()
            except OSError:
                pass

# --- Lean file reading ---

def read_lean_module(module_name):
    """Read a .lean source file from the grothendieck_lean project.

    module_name: e.g. 'CategoryAndSites' -> reads Grothendieck/CategoryAndSites.lean
    Returns the file content as a string.
    """
    path = WIN_LEAN_PROJECT / 'Grothendieck' / f'{module_name}.lean'
    if not path.exists():
        return f'[FICHIER INTROUVABLE] {path}'
    return path.read_text(encoding='utf-8')

def display_lean_module(module_name, max_lines=None, highlight=None):
    """Display a .lean source file with line numbers.

    max_lines: if set, only show the first N lines
    highlight: list of line numbers to mark with '>>>' (1-indexed)
    """
    content = read_lean_module(module_name)
    if content.startswith('[FICHIER INTROUVABLE]'):
        print(content)
        return
    lines = content.splitlines()
    if max_lines:
        lines = lines[:max_lines]
    highlight = set(highlight or [])
    print(f'--- Grothendieck/{module_name}.lean ---')
    for i, line in enumerate(lines, 1):
        marker = ' >>>' if i in highlight else '    '
        print(f'{marker} {i:>3d} | {line}')
    total = len(content.splitlines())
    if max_lines and total > max_lines:
        print(f'    ... ({total - max_lines} lignes restantes sur {total} total)')
    print(f'--- fin ({total} lignes) ---')

# --- Lean snippet execution ---

def run_lean(snippet, timeout_s=300):
    """Run a Lean snippet against the grothendieck_lean project using lake env lean.

    The snippet is written to a temp file and executed with the project's Lake env.
    Returns combined stdout+stderr.
    """
    snippet = textwrap.dedent(snippet).strip() + '\n'
    write_cmd = f"cat > /tmp/lean13b_snippet.lean << 'LEAN_EOF'\n{snippet}LEAN_EOF"
    lean_cmd = f'cd {LEAN_PROJECT} && lake env lean /tmp/lean13b_snippet.lean 2>&1'
    full_cmd = f'{write_cmd}\n{lean_cmd}'
    rc, out, err = wsl(full_cmd, timeout=timeout_s)
    if rc == -1:
        return f'Snippet Lean en attente du build Lake (timeout apres {timeout_s}s). '\
               f'Lancez: wsl -d Ubuntu -- bash -lc "cd {LEAN_PROJECT} && lake build Grothendieck"'
    return (out or '') + (err or '')

# --- Module inventory ---

GROTHENDIECK_MODULES = {
    'CategoryAndSites': 'Part 1: Sieves, topologies, 3 axioms',
    'SchemesTour': 'Part 2: Scheme, Spec, Gamma',
    'ZariskiSite': 'Part 3: Zariski pretopology, bridge theorem',
    'MathlibMap': 'Part 4: #check index Mathlib',
    'Calibration': 'Part 5: 4 micro-preuves P1-P4',
    'SieveLattice': 'Part 6: Pullback identities',
    'SheafBasics': 'Part 7: Sheaves, sheaf condition',
    'SieveOps': 'Part 8: Sieve lattice operations',
    'CoverageGen': 'Part 9: Coverage generators',
    'CanonicalProps': 'Part 10: Canonical topology properties',
    'SieveGenerate': 'Part 11: Sieve generation',
    'DenseTopology': 'Part 12: Dense topology',
    'Sheafification': 'Part 13: Sheafification functor (Mathlib bridge)',
    'LeftExact': 'Part 14: Left exactness of sheafification',
    'SitePoints': 'Part 15: Points of a site',
    'Subcanonical': 'Part 16: Subcanonical topologies, Yoneda sheaves',
    'SheafHom': 'Part 17: Sheaf hom, internal hom',
    'ConstantSheaf': 'Part 18: Constant sheaf, adjunction',
    'Conservative': 'Part 19: Conservative families of points',
    'SheafCohomology/Basic': 'Part 20: Ext-based sheaf cohomology H^n',
    'MayerVietorisSquare': 'Part 21: Mayer-Vietoris squares',
    'SheafCohomology/MayerVietoris': 'Part 22: Mayer-Vietoris long exact sequence',
    'SheafCohomology/Cech': 'Part 23: Cech cohomology complex',
}

# Verify project is accessible
assert (WIN_LEAN_PROJECT / 'lakefile.lean').exists(), 'grothendieck_lean/lakefile.lean not found'
print(f'Setup OK : grothendieck_lean project detecte a {WIN_LEAN_PROJECT}')
print(f'  WSL path : {LEAN_PROJECT}')
print(f'  {len(GROTHENDIECK_MODULES)} modules Grothendieck disponibles')

Setup OK : grothendieck_lean project detecte a <repo>MyIA.AI.Notebooks\SymbolicAI\Lean\grothendieck_lean
  WSL path : <repo>MyIA.AI.Notebooks/SymbolicAI/Lean/grothendieck_lean
  23 modules Grothendieck disponibles


## 1. Le projet Lake `grothendieck_lean/`

Le projet `grothendieck_lean/` est un **workspace Lake** dedie a l'exploration pedagogique du langage mathematique de Grothendieck dans Mathlib 4. Il contient 23 modules sous le namespace `Grothendieck` (6 modules pedagogiques detailles dans cet atelier + 17 modules avances couvrant faisceaux, cohomologie et Cech).

### Architecture du projet

```
grothendieck_lean/
  lakefile.lean          -- dependance mathlib4
  lean-toolchain         -- leanprover/lean4:v4.30.0-rc2
  Grothendieck.lean      -- module racine (importe les 23 sous-modules)
  Grothendieck/
    CategoryAndSites.lean  -- Part 1: Cribles, topologies, 3 axiomes
    SchemesTour.lean       -- Part 2: Scheme, Spec, Gamma
    ZariskiSite.lean       -- Part 3: Pretopologie de Zariski, bridge theorem
    MathlibMap.lean        -- Part 4: #check living index
    Calibration.lean       -- Part 5: 4 micro-preuves P1-P4
    SieveLattice.lean      -- Part 6: Identites de pullback
```

**Convention** : tous les modules utilisent le namespace `Grothendieck` et 0 sorry en code de production.

In [2]:
# Vue d'ensemble du projet grothendieck_lean
print('Structure du projet grothendieck_lean/')
print('=' * 60)
print()

# Module racine
root = WIN_LEAN_PROJECT / 'Grothendieck.lean'
print(f'Grothendieck.lean (racine) : {len(root.read_text(encoding="utf-8").splitlines())} lignes')
print()

# Sous-modules
total_lines = 0
total_sorry = 0
for mod_name, desc in GROTHENDIECK_MODULES.items():
    content = read_lean_module(mod_name)
    lines = len(content.splitlines())
    # Compter sorry (hors commentaires)
    stripped = re.sub(r'/-.*?-/', '', content, flags=re.DOTALL)
    stripped = re.sub(r'--.*$', '', stripped, flags=re.MULTILINE)
    sorry_count = sum(1 for l in stripped.splitlines() if 'sorry' in l.strip())
    total_lines += lines
    total_sorry += sorry_count
    print(f'  Grothendieck/{mod_name:<20s} {lines:>4d} lignes  sorry={sorry_count}  -- {desc}')

print('-' * 60)
print(f'  TOTAL                       {total_lines:>4d} lignes  sorry={total_sorry}')
print()
print(f'Toolchain : {(WIN_LEAN_PROJECT / "lean-toolchain").read_text(encoding="utf-8").strip()}')
print()
print('Module racine (imports) :')
display_lean_module('../Grothendieck', max_lines=50)

Structure du projet grothendieck_lean/

Grothendieck.lean (racine) : 43 lignes

  Grothendieck/CategoryAndSites      106 lignes  sorry=0  -- Part 1: Sieves, topologies, 3 axioms
  Grothendieck/SchemesTour            79 lignes  sorry=0  -- Part 2: Scheme, Spec, Gamma
  Grothendieck/ZariskiSite            84 lignes  sorry=0  -- Part 3: Zariski pretopology, bridge theorem
  Grothendieck/MathlibMap             90 lignes  sorry=0  -- Part 4: #check index Mathlib
  Grothendieck/Calibration            80 lignes  sorry=0  -- Part 5: 4 micro-preuves P1-P4
  Grothendieck/SieveLattice           88 lignes  sorry=0  -- Part 6: Pullback identities
  Grothendieck/SheafBasics           128 lignes  sorry=0  -- Part 7: Sheaves, sheaf condition
  Grothendieck/SieveOps              124 lignes  sorry=0  -- Part 8: Sieve lattice operations
  Grothendieck/CoverageGen           148 lignes  sorry=0  -- Part 9: Coverage generators
  Grothendieck/CanonicalProps        133 lignes  sorry=0  -- Part 10: Canonical t

### Interpretation : architecture du projet

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Modules | 23 | 6 pedagogiques + 17 avances (faisceaux, cohomologie, Cech) |
| Lignes totales | ~3182 | Projet pedagogique etendu (Parts 1-23) |
| sorry | 0 | Toutes les preuves sont completes |
| Toolchain | v4.30.0-rc2 | Version recente de Lean 4 |

**Points cles** :
1. Le module racine `Grothendieck.lean` importe les 23 sous-modules. Un `lake build Grothendieck` compile tout.
2. Chaque module est autonome (imports Mathlib directs, pas de dependances inter-modules autres que via Mathlib).
3. Les 4 micro-preuves de `Calibration.lean` sont les seules preuves non triviales ; le reste est des `#check` et des `example`/`theorem` a une ligne.

## 2. CategoryAndSites : cribles et topologies de Grothendieck

Ce module introduit les **cribles** (`Sieve X`) et les **topologies de Grothendieck** (`GrothendieckTopology C`). Ce sont les fondations sur lesquelles tout le reste repose.

Un **crible** sur un objet `X` est un sous-foncteur de $\text{Hom}(-, X)$ (le plongement de Yoneda). Une **topologie de Grothendieck** sur une categorie `C` assigne a chaque objet `X` une collection de cribles couvrants, soumise a trois axiomes :

1. **Axiome d'identite** : le crible maximal $\top$ couvre toujours.
2. **Stabilite par pullback** : les cribles couvrants sont stables par image inverse.
3. **Transitivite** : raffiner un crible couvrant par des couvrants donne un couvrant.

Le module montre aussi que les topologies de Grothendieck forment un **treillis complet** (avec $\bot = $ `trivial`, $\top = $ `discrete`).

In [3]:
# Affichage complet du module CategoryAndSites
display_lean_module('CategoryAndSites')

--- Grothendieck/CategoryAndSites.lean ---
       1 | /-
       2 | Grothendieck tribute — Part 1: Categories, Sieves, and Grothendieck Topologies
       3 | Alexandre Grothendieck (1928-2014).
       4 | 
       5 | Grothendieck revolutionized algebraic geometry by replacing topological spaces
       6 | with categories equipped with a "topology" defined by covering sieves. This file
       7 | tours the Mathlib 4 formalization of these concepts.
       8 | 
       9 | The key insight: a Grothendieck topology on a category C assigns to each object X
      10 | a collection of "covering sieves" satisfying three axioms:
      11 |   1. The maximal sieve always covers (stability under identity)
      12 |   2. Covering sieves are stable under pullback (locality)
      13 |   3. If S covers X and R pulls back to a covering sieve along every arrow in S,
      14 |      then R covers X (transitivity)
      15 | 
      16 | Epic #1646. All `sorry`s eliminated at creation.
      17 | -/
     

### Interpretation : les trois axiomes en Lean

Le module definit trois theoremes qui correspondent exactement aux trois axiomes de SGA 4 :

| Theoreme Lean | Axiome SGA 4 | Enonce intuitif |
|---------------|-------------|-----------------|
| `top_covers` | Identite | $\top \in J(X)$ toujours |
| `pullback_cover` | Stabilite | $S \in J(X) \Rightarrow S.\text{pullback}(f) \in J(Y)$ |
| `transitivity` | Transitivite | Si $S$ couvre et chaque fleche de $S$ tire en un couvrant, alors le raffinement couvre |

**Observation pedagogique** : chaque theoreme est une simple reformulation d'un champ de la structure `GrothendieckTopology` :
- `J.top_mem X` pour l'axiome 1
- `J.pullback_stable f hS` pour l'axiome 2
- `J.transitive hS R hR` pour l'axiome 3

Le lecteur peut verifier que la definition Mathlib epouse exactement celle de SGA 4, exposant I.

In [4]:
# Verification interactive : le treillis des cribles
# Ce snippet verifie que Sieve X est un CompleteLattice (meta-prop qui n'affiche rien,
# mais la compilation reussie confirme l'instance).
snippet = """
import Mathlib.CategoryTheory.Sites.Grothendieck

#check @instCompleteLatticeSieve
-- Signature : {C : Type u_1} -> [inst : Category C] -> {X : C} -> CompleteLattice (Sieve X)

#check @GrothendieckTopology.trivial
#check @GrothendieckTopology.discrete
#check @GrothendieckTopology.dense
"""

print("Execution du snippet Lean (verification CompleteLattice + topologies extremes)...")
result = run_lean(snippet, timeout_s=300)
if 'does not exist' in result:
    print("Snippet Lean en attente du build Lake (fichiers .olean manquants).")
    print("Pour activer les snippets interactifs, lancez d'abord :")
    print(f'  wsl -d Ubuntu -- bash -lc "cd {LEAN_PROJECT} && lake build Grothendieck"')
    print()
    print("Resultat attendu (une fois le build termine) :")
    print("  #check @instCompleteLatticeSieve -> ... CompleteLattice (Sieve X)")
    print("  #check @GrothendieckTopology.trivial -> GrothendieckTopology C")
    print("  #check @GrothendieckTopology.discrete -> GrothendieckTopology C")
    print("  #check @GrothendieckTopology.dense    -> GrothendieckTopology C")
elif 'TIMEOUT' in result or 'en attente' in result:
    print(result)
else:
    lines = [l for l in result.splitlines() if not l.startswith('[')]
    for l in lines:
        if l.strip():
            print(l)

Execution du snippet Lean (verification CompleteLattice + topologies extremes)...


/tmp/lean13b_snippet.lean:3:8: error(lean.unknownIdentifier): Unknown identifier `instCompleteLatticeSieve`
/tmp/lean13b_snippet.lean:6:8: error(lean.unknownIdentifier): Unknown identifier `GrothendieckTopology.trivial`
/tmp/lean13b_snippet.lean:7:8: error(lean.unknownIdentifier): Unknown identifier `GrothendieckTopology.discrete`
/tmp/lean13b_snippet.lean:8:8: error(lean.unknownIdentifier): Unknown identifier `GrothendieckTopology.dense`


## 3. SchemesTour : schemas, Spec et sections globales

Le module `SchemesTour` presente les trois piliers de la geometrie algebrique grothendieckienne dans Mathlib :

- **`Scheme`** : le type des schemas (espaces localement annees localement isomorphes a `Spec R`).
- **`Scheme.Spec`** : le foncteur spectre, des anneaux vers les schemas.
- **`Scheme.\u0393`** (Gamma) : le foncteur sections globales, des schemas vers les anneaux.

L'adjonction Spec-Gamma est le coeur de la geometrie algebrique : $\text{Spec} \dashv \Gamma$.

In [5]:
# Affichage complet du module SchemesTour
display_lean_module('SchemesTour')

--- Grothendieck/SchemesTour.lean ---
       1 | /-
       2 | Grothendieck tribute — Part 2: Schemes
       3 | Alexandre Grothendieck (1928-2014).
       4 | 
       5 | Grothendieck's most transformative idea: replace varieties by *schemes* — locally
       6 | ringed spaces that are locally affine (isomorphic to Spec R for a commutative ring R).
       7 | This gives a single framework for arithmetic and geometry.
       8 | 
       9 | Mathlib 4 formalizes schemes as `AlgebraicGeometry.Scheme`, extending
      10 | `LocallyRingedSpace` with the local-affineness condition.
      11 | 
      12 | Epic #1646. All `sorry`s eliminated at creation.
      13 | -/
      14 | 
      15 | import Mathlib.AlgebraicGeometry.Scheme
      16 | 
      17 | namespace Grothendieck
      18 | 
      19 | open AlgebraicGeometry CategoryTheory
      20 | 
      21 | /-!
      22 | ## The type of schemes
      23 | 
      24 | `Scheme` is the type of schemes. It carries a category structure.
      25 |

### Interpretation : Spec et Gamma

Les constructions cles de ce module :

| Construction | Type Lean | Lecture |
|-------------|-----------|--------|
| `Scheme` | `Type (u+1)` | le type des schemas |
| `Scheme.Spec` | `CommRingCat^op ⥤ Scheme` | foncteur spectre (anneau -> espace) |
| `Scheme.Γ.obj (Opposite.op X)` | `CommRingCat` | sections globales d'un schema |
| `Scheme.forgetToTop` | `Scheme ⥤ TopCat` | foncteur d'oubli vers les espaces topologiques |
| `Scheme.homeoOfIso i` | `X ≃ₜ Y` | un isomorphisme de schemas induit un homeomorphisme |

**Point pedagogique** : le module montre la **dualite algebre-geometrie** au coeur du programme de Grothendieck. `Spec` transforme un anneau en espace ; `Γ` transforme un espace en anneau. Pour les schemas affines, ces foncteurs sont des equivalences inverses.

In [6]:
# Verification interactive : Scheme et ses foncteurs
snippet = """
import Mathlib.AlgebraicGeometry.Scheme

open AlgebraicGeometry CategoryTheory

-- Le type des schemas
#check Scheme

-- Spec : des anneaux vers les schemas
#check @Scheme.Spec

-- Sections globales
#check @Scheme.Γ

-- Un schema a des sections globales
example (X : Scheme) : CommRingCat := Scheme.Γ.obj (Opposite.op X)
"""

print("Execution du snippet Lean (Scheme, Spec, Gamma)...")
result = run_lean(snippet, timeout_s=300)
if 'does not exist' in result:
    print("Snippet Lean en attente du build Lake (fichiers .olean manquants).")
    print("Pour activer les snippets interactifs, lancez d'abord :")
    print(f'  wsl -d Ubuntu -- bash -lc "cd {LEAN_PROJECT} && lake build Grothendieck"')
    print()
    print("Resultat attendu :")
    print("  #check Scheme       -> Type (u+1)")
    print("  #check @Scheme.Spec -> CommRingCat^op ⥤ Scheme")
    print("  #check @Scheme.Γ    -> Scheme^op ⥤ CommRingCat")
elif 'TIMEOUT' in result or 'en attente' in result:
    print(result)
else:
    lines = [l for l in result.splitlines() if not l.startswith('[')]
    for l in lines:
        if l.strip():
            print(l)

Execution du snippet Lean (Scheme, Spec, Gamma)...


AlgebraicGeometry.Scheme.{u_1} : Type (u_1 + 1)
Scheme.Spec : CommRingCatᵒᵖ ⥤ Scheme
Scheme.Γ : Schemeᵒᵖ ⥤ CommRingCat


## 4. ZariskiSite : la topologie de Zariski comme topologie de Grothendieck

Le module `ZariskiSite` presente l'exemple le plus important de topologie de Grothendieck issue de la geometrie algebrique. La **topologie de Zariski** sur la categorie des schemas est definie via une **pretopologie** (familles d'immersions ouvertes recouvrantes), puis montee en topologie de Grothendieck.

Le **theoreme pont** (`zariskiTopology_eq`) etablit que la topologie de Grothendieck engendree par la pretopologie de Zariski est bien la topologie de Zariski.

In [7]:
# Affichage complet du module ZariskiSite
display_lean_module('ZariskiSite')

--- Grothendieck/ZariskiSite.lean ---
       1 | /-
       2 | Grothendieck tribute — Part 3: The Zariski site
       3 | Alexandre Grothendieck (1928-2014).
       4 | 
       5 | The Zariski topology on the category of schemes is the foundational example
       6 | of a Grothendieck topology arising from algebraic geometry. A family of
       7 | morphisms {U_i → X} is a Zariski cover iff the U_i are open immersions
       8 | that jointly cover X.
       9 | 
      10 | Mathlib 4 formalizes this as `Scheme.zariskiTopology`, derived from the
      11 | pretopology of open immersions. The key bridge theorem is
      12 | `zariskiTopology_eq`: the Grothendieck topology generated by the Zariski
      13 | pretopology equals the Zariski topology.
      14 | 
      15 | Epic #1646. All `sorry`s eliminated at creation.
      16 | -/
      17 | 
      18 | import Mathlib.AlgebraicGeometry.Sites.BigZariski
      19 | 
      20 | namespace Grothendieck
      21 | 
      22 | open AlgebraicGeo

### Interpretation : du concret a l'abstrait

Le module illustre la progression **concret -> abstrait** typique de la methode grothendieckienne :

1. **Pretopologie** (`Scheme.zariskiPretopology`) : definition concrete par familles de morphismes recouvrants.
2. **Topologie de Grothendieck** (`Scheme.zariskiTopology`) : definition abstraite par axiomes sur les cribles.
3. **Theoreme pont** (`zariskiTopology_eq`) : les deux points de vue coincident.

| Propriete | Enonce Lean |
|-----------|------------|
| Sous-canonique | `Scheme.zariskiTopology.Subcanonical` (tout representable est un faisceau) |
| Continuite de l'oubli | `Scheme.forgetToTop.IsContinuous` (l'oubli est continu) |

La propriete **sous-canonique** signifie que les schemas eux-memes "se recollent" pour la topologie de Zariski -- consequence non triviale du lemme de Yoneda.

In [8]:
# Verification interactive : la pretopologie et la topologie de Zariski
snippet = """
import Mathlib.AlgebraicGeometry.Sites.BigZariski

open AlgebraicGeometry CategoryTheory

-- Pretopologie de Zariski
#check @Scheme.zariskiPretopology

-- Topologie de Zariski
#check @Scheme.zariskiTopology

-- Le theoreme pont
#check @Scheme.zariskiTopology_eq
"""

print("Execution du snippet Lean (Zariski pretopology + topology)...")
result = run_lean(snippet, timeout_s=300)
if 'does not exist' in result:
    print("Snippet Lean en attente du build Lake (fichiers .olean manquants).")
    print("Pour activer les snippets interactifs, lancez d'abord :")
    print(f'  wsl -d Ubuntu -- bash -lc "cd {LEAN_PROJECT} && lake build Grothendieck"')
    print()
    print("Resultat attendu :")
    print("  #check @Scheme.zariskiPretopology -> Pretopology Scheme")
    print("  #check @Scheme.zariskiTopology     -> GrothendieckTopology Scheme")
    print("  #check @Scheme.zariskiTopology_eq  -> zariskiTopology = zariskiPretopology.toGrothendieck")
elif 'TIMEOUT' in result or 'en attente' in result:
    print(result)
else:
    lines = [l for l in result.splitlines() if not l.startswith('[')]
    for l in lines:
        if l.strip():
            print(l)

Execution du snippet Lean (Zariski pretopology + topology)...


Scheme.zariskiPretopology : Pretopology Scheme
Scheme.zariskiTopology : GrothendieckTopology Scheme
Scheme.zariskiTopology_eq : Scheme.zariskiTopology = Scheme.zariskiPretopology.toGrothendieck


## 5. Calibration : les 4 micro-preuves (P1-P4)

Le module `Calibration` est le coeur **preuve** du projet. Il contient 4 theoremes qui exercice des strategies de preuve differentes :

| Preuve | Strategie | Enonce |
|--------|----------|--------|
| P1 | Reecriture + `bot_le` | `trivial C ≤ discrete C` (ordre du treillis) |
| P2 | Extensionnalite + `simp` | `Sieve.pullback f ⊤ = ⊤` |
| P3 | `exact` (lemme existant) | `zariskiTopology = zariskiPretopology.toGrothendieck` |
| P4 | `exact` (lemme general) | Tout prefaisceau est un faisceau pour la topologie triviale |

Chaque preuve est courte (1-3 lignes de tactiques) mais illustre un pattern recurrent en formalisation Mathlib.

In [9]:
# Affichage complet du module Calibration
display_lean_module('Calibration')

--- Grothendieck/Calibration.lean ---
       1 | /-
       2 | Grothendieck tribute — Part 5: Calibration targets for the prover harness
       3 | Alexandre Grothendieck (1928-2014).
       4 | 
       5 | Micro-proof targets for the harness calibration ladder (Epic #1453).
       6 | These are deliberately simple facts about Grothendieck topologies that
       7 | exercise different proof strategies:
       8 | 
       9 |   - P1 (closed-eval): trivial ≤ discrete (lattice order)
      10 |   - P2 (case-decomposition): Sieve.pullback of ⊤ is ⊤
      11 |   - P3 (rewriting): zariskiTopology_eq bridge theorem
      12 |   - P4 (integration): every presheaf is a sheaf for the trivial topology
      13 | 
      14 | Epic #1646, #1453. All `sorry`s eliminated at creation.
      15 | -/
      16 | 
      17 | import Mathlib.CategoryTheory.Sites.Grothendieck
      18 | import Mathlib.AlgebraicGeometry.Sites.BigZariski
      19 | 
      20 | namespace Grothendieck
      21 | 
      22 | open 

### Interpretation : analyse des tactiques

**P1 : `trivial_le_discrete`**
```lean
rw [GrothendieckTopology.trivial_eq_bot, GrothendieckTopology.discrete_eq_top]
exact bot_le
```
Strategie : reecrire `trivial = ⊥` et `discrete = ⊤`, puis appliquer le lemme general `bot_le` du treillis. Le prover doit découvrir les deux equations de reecriture.

**P2 : `pullback_top`**
```lean
ext Z g
simp [Sieve.pullback]
```
Strategie : extensionnalite des cribles (deux cribles sont egaux ssi ils ont les memes fleches), puis `simp` avec la definition de `Sieve.pullback`. La tactique `ext` decompose l'egalite fonctionnelle.

**P3 : `zariski_eq_pretopology`**
```lean
exact Scheme.zariskiTopology_eq
```
Strategie : appel direct au lemme de Mathlib. Le defi pour le prover est de **localiser** le bon lemme dans la bibliotheque.

**P4 : `isSheaf_trivial`**
```lean
exact Presieve.isSheaf_bot
```
Strategie : un lemme general de Mathlib qui dit que la topologie ⊥ rend tout prefaisceau faisceau. Le prover doit identifier ce lemme.

> **Note** : ces 4 preuves sont les cibles de calibration pour le harness de preuve automatique (Epic #1453). Elles servent de "tests unitaires" pour verifier que le prover sait utiliser differentes strategies.

## 6. SieveLattice : le treillis des cribles et le pullback

Le module `SieveLattice` complete `CategoryAndSites` en etudiant les **identites du pullback** dans le treillis des cribles. Le pullback d'un crible le long d'un morphisme est l'operation fondamentale qui permet de definir la stabilite des topologies de Grothendieck.

Les 4 identites sont :

1. **`pullback_id`** : $S.\text{pullback}(\mathbf{1}_X) = S$ (pullback le long de l'identite = identite)
2. **`pullback_pullback`** : $S.\text{pullback}(f).\text{pullback}(g) = S.\text{pullback}(g \circ f)$ (contravariance)
3. **`pullback_bot`** : $\bot.\text{pullback}(f) = \bot$ (pullback du vide = vide)
4. **`pullback_monotone`** : $S \leq T \Rightarrow S.\text{pullback}(f) \leq T.\text{pullback}(f)$ (monotonie)

Ces identites, avec `pullback_top` (Calibration P2), forment un ensemble complet de proprietes structurelles du pullback.

In [10]:
# Affichage complet du module SieveLattice
display_lean_module('SieveLattice')

--- Grothendieck/SieveLattice.lean ---
       1 | /-
       2 | Grothendieck tribute — Part 6: Sieve pullback identities and lattice laws
       3 | Alexandre Grothendieck (1928-2014).
       4 | 
       5 | Phase 2 extension (#2159, Epic #2162).
       6 | 
       7 | Part 1 (CategoryAndSites.lean) introduces sieves, the three axioms,
       8 | and the complete lattice on Sieve X. This module records the basic
       9 | identities of pullback along morphisms:
      10 | 
      11 |   - Pullback along the identity is the identity (pullback_id).
      12 |   - Pullback composes contravariantly (pullback_pullback).
      13 |   - Pullback of the bottom sieve is bottom (pullback_bot).
      14 |   - Pullback is monotone in the sieve (pullback_monotone).
      15 | 
      16 | These complete the picture started by Part 5 calibration P2
      17 | (pullback_top in Calibration.lean), and pave the way for Phase 3
      18 | work on sieve generation and sheafification.
      19 | 
      20 |

### Interpretation : structure du treillis des cribles

Les 4 identites se lisent comme les **axiomes d'un foncteur contravariant** du treillis des cribles :

| Identite | Type | Analogie categorique |
|----------|------|---------------------|
| `pullback_id` | $S.\text{pb}(\mathbf{1}) = S$ | Preservation de l'identite |
| `pullback_pullback` | $S.\text{pb}(f).\text{pb}(g) = S.\text{pb}(g \circ f)$ | Contravariance de la composition |
| `pullback_bot` | $\bot.\text{pb}(f) = \bot$ | Preservation du minimum |
| `pullback_monotone` | $S \leq T \Rightarrow S.\text{pb}(f) \leq T.\text{pb}(f)$ | Monotonie (foncteur de treillis) |

**Toutes les preuves** suivent le meme pattern : `ext` pour l'extensionantalite, puis `simp` avec la definition de `Sieve.pullback`. C'est un pattern systematique en Mathlib pour les egalites de sous-foncteurs.

La seule exception est `pullback_monotone`, qui utilise `intro Z g hg` + `simp` + `apply hST` (argument d'ordre).

## 7. MathlibMap : l'index vivant

Le module `MathlibMap` est un **catalogue** de `#check` qui verifient que chaque definition cle du langage grothendieckien est accessible dans Mathlib. Il sert de carte de reference et de test de non-regression.

Le module couvre 5 domaines :
1. Fondations categoriques (Yoneda)
2. Cribles et pre-cribles
3. Topologies de Grothendieck
4. Faisceaux
5. Geometrie algebrique (Scheme, Spec, Gamma)

In [11]:
# Affichage complet du module MathlibMap
display_lean_module('MathlibMap')

--- Grothendieck/MathlibMap.lean ---
       1 | /-
       2 | Grothendieck tribute — Part 4: Mathlib Map
       3 | Alexandre Grothendieck (1928-2014).
       4 | 
       5 | A living index of what Mathlib 4 provides from Grothendieck's mathematical
       6 | language. Each `#check` verifies that the definition exists and is accessible
       7 | from the current imports.
       8 | 
       9 | Epic #1646. All `sorry`s eliminated at creation.
      10 | -/
      11 | 
      12 | import Mathlib.CategoryTheory.Sites.Grothendieck
      13 | import Mathlib.CategoryTheory.Sites.SheafOfTypes
      14 | import Mathlib.AlgebraicGeometry.Scheme
      15 | import Mathlib.Topology.Sheaves.Sheaf
      16 | 
      17 | /-!
      18 | ## Category theory foundations (Grothendieck's legacy)
      19 | 
      20 | Grothendieck made category theory the language of algebraic geometry.
      21 | Mathlib 4 has a rich category theory library built on these ideas.
      22 | -/
      23 | 
      24 | -- Th

### Interpretation : ce que Mathlib a (et n'a pas encore)

Le module `MathlibMap` sert de **sonde** : chaque `#check` confirme qu'une definition est presente et accessible. La section finale liste explicitement ce qui **manque** :

**Disponible dans Mathlib** (verifie par #check) :

| Domaine | Definitions |
|---------|------------|
| Categories | `yoneda`, `coyoneda`, `Functor` |
| Cribles | `Sieve`, `Presieve`, `Sieve.pullback`, `Sieve.arrows` |
| Topologies | `GrothendieckTopology`, `trivial`, `discrete`, `dense` |
| Faisceaux | `Presieve.IsSheaf`, `Presieve.IsSeparated`, `TopCat.Sheaf` |
| Geometrie | `Scheme`, `Scheme.Spec`, `Scheme.Γ`, `Scheme.forgetToTop` |

**Pas encore dans Mathlib** (2026) :
- Cohomologie etale
- Motifs
- Six operations
- Grothendieck-Riemann-Roch
- Dualite de Grothendieck
- Geometrie anabelienne

In [12]:
# Comptage systematique des #check dans MathlibMap
content = read_lean_module('MathlibMap')
check_lines = [l.strip() for l in content.splitlines() if l.strip().startswith('#check')]
print(f'MathlibMap : {len(check_lines)} verifications #check')
print()
for i, line in enumerate(check_lines, 1):
    # Extraire le nom court
    name = line.replace('#check @', '').replace('#check ', '')
    print(f'  {i:>2d}. {name}')

MathlibMap : 18 verifications #check

   1. CategoryTheory.yoneda            -- C ⥤ (Cᵒᵖ ⥤ Type v)
   2. CategoryTheory.coyoneda          -- (Cᵒᵖ ⥤ Type v) ⥤ C
   3. CategoryTheory.Presieve          -- Presieve X
   4. CategoryTheory.Sieve             -- Sieve X (subfunctor of yoneda.obj X)
   5. CategoryTheory.Sieve.pullback    -- pullback a sieve along a morphism
   6. CategoryTheory.Sieve.arrows      -- the underlying presieve
   7. CategoryTheory.GrothendieckTopology          -- the topology structure
   8. CategoryTheory.GrothendieckTopology.trivial  -- coarsest topology
   9. CategoryTheory.GrothendieckTopology.discrete -- finest topology
  10. CategoryTheory.GrothendieckTopology.dense    -- dense topology
  11. CategoryTheory.Presieve.IsSheaf  -- sheaf condition for Type-valued presheaves
  12. CategoryTheory.Presieve.IsSeparated  -- separated presheaf
  13. TopCat.Sheaf                     -- bundled sheaf on a topological space
  14. Scheme                   -- the type of sch

## 8. Exemples guidés

Les exercices ci-dessous ont ete realises par des etudiants. Chaque solution est presentee comme un **exemple guide** complet (solution fonctionnelle), suivi de l'exercice original (stub a completer). Les deux cohabitent : l'exemple montre la solution, l'exercice invite a reproduire ou varier.

### Exercice 1 : explorer un concept categorique non couvert

**Objectif** : ecrire un snippet Lean qui verifie l'existence d'une construction categorique liee a Grothendieck mais absente de `MathlibMap`.

**Indice** : cherchez dans `Mathlib.CategoryTheory.Limits` ou `Mathlib.CategoryTheory.Adjunction` un concept non present dans le module (par exemple, `CategoryTheory.Limits.HasEqualizers`, `CategoryTheory.Adjunction.leftAdjoint`...).

**Etapes** :
1. Choisir un concept categorique (limites, adjonctions, transformees naturelles...).
2. Construire un snippet avec l'import approprie et un `#check`.
3. Executer avec `run_lean()` ou simplement ecrire le snippet.

In [13]:
# Exemple guide 1 : explorer un concept categorique non couvert par MathlibMap
# Corrige — rendu PR #2677 (@starsamk)
# L'etudiant a choisi HasEqualizers comme concept non couvert par MathlibMap

snippet_ex1_corrige = """
import Mathlib.CategoryTheory.Limits.Shapes.Equalizers

#check @CategoryTheory.Limits.HasEqualizers
"""

try:
    resultat_ex1_corrige = run_lean(snippet_ex1_corrige, timeout_s=300)
    print(resultat_ex1_corrige)
except Exception as e:
    print(f"Exemple guide 1 : snippet Lean valide (HasEqualizers). Execution WSL non disponible : {e}")
    print("Resultat attendu : #check @CategoryTheory.Limits.HasEqualizers -> Prop")

CategoryTheory.Limits.HasEqualizers : (C : Type u_2) → [CategoryTheory.Category.{u_1, u_2} C] → Prop



### Exemple guide 2 : prouver une identite sur Sieve.pullback

**Objectif** : ecrire et prouver un theoreme simple sur `Sieve.pullback`, inspire de `SieveLattice.lean`.

**Solution etudiante** : `pullback_top_variant` prouve que le pullback du crible maximal est maximal, en utilisant `ext Z g` + `simp [Sieve.pullback]` (pattern identique a Calibration P2).

In [14]:
# Exemple guide 2 : prouver une identite sur Sieve.pullback
# Corrige — rendu PR #2677 (@starsamk)
# Preuve : ext Z g / simp [Sieve.pullback] (pattern SieveLattice)

snippet_ex2_corrige = """
import Mathlib.CategoryTheory.Sites.Grothendieck

open CategoryTheory

theorem pullback_top_variant {C : Type*} [Category C] {X Y : C} (f : Y ⟶ X) :
    (Sieve.pullback f ⊤ : Sieve Y) = ⊤ := by
  ext Z g
  simp [Sieve.pullback]
"""

try:
    resultat_ex2_corrige = run_lean(snippet_ex2_corrige, timeout_s=300)
    print(resultat_ex2_corrige)
except Exception as e:
    print(f"Exemple guide 2 : snippet Lean valide (pullback_top_variant). Execution WSL non disponible : {e}")
    print("Resultat attendu : theorem pullback_top_variant : no errors")

### Exemple guide 3 : ajouter une micro-preuve a Calibration

**Objectif** : ecrire une nouvelle micro-preuve dans le style de `Calibration.lean`, en utilisant une tactique differente de P1-P4.

**Solution etudiante** : `bot_le_topology` prouve que la topologie triviale est inferieure ou egale a toute topologie de Grothendieck, en utilisant `rw [GrothendieckTopology.trivial_eq_bot]` + `exact bot_le` (pattern similaire a P1).

In [15]:
# Exemple guide 3 : ajouter une micro-preuve a Calibration
# Corrige — rendu PR #2677 (@starsamk)
# Preuve : rw [GrothendieckTopology.trivial_eq_bot] + exact bot_le (pattern Calibration P1)

snippet_ex3_corrige = """
import Mathlib.CategoryTheory.Sites.Grothendieck

open CategoryTheory

-- P5 candidate : le treillis des topologies est ordonne
theorem bot_le_topology {C : Type*} [Category C] (J : GrothendieckTopology C) :
    GrothendieckTopology.trivial C ≤ J := by
  rw [GrothendieckTopology.trivial_eq_bot]
  exact bot_le
"""

try:
    resultat_ex3_corrige = run_lean(snippet_ex3_corrige, timeout_s=300)
    print(resultat_ex3_corrige)
except Exception as e:
    print(f"Exemple guide 3 : snippet Lean valide (bot_le_topology). Execution WSL non disponible : {e}")
    print("Resultat attendu : theorem bot_le_topology : no errors")

## 9. Exercices

Les exercices suivants sont des **stubs a completer**. Ils reprennent les enonces des exemples guidés de la section 8, mais avec un squelette a completer par l'etudiant.

### Exercice 1 : explorer un concept categorique non couvert

**Objectif** : choisir un concept categorique lie a Grothendieck et construire un snippet `#check`.

**Indice** : `CategoryTheory.Limits.HasEqualizers`, `CategoryTheory.Adjunction`, `CategoryTheory.Monoidal`...

**Etapes** :
1. Identifier un concept categorique lie a Grothendieck.
2. Ecrire le snippet avec l'import et le `#check`.
3. Executer avec `run_lean(snippet, timeout_s=300)`.

In [16]:
# Exercice 1 : explorer un concept categorique non couvert par MathlibMap
# TODO etudiant : choisir un concept et construire un snippet #check
# Etape 1 : identifier un concept categorique lie a Grothendieck
#   Suggestions : CategoryTheory.Limits.HasEqualizers, CategoryTheory.Adjunction,
#   CategoryTheory.Monoidal, CategoryTheory.Comma...
# Etape 2 : ecrire le snippet avec l'import et le #check
# Etape 3 : executer avec run_lean(snippet, timeout_s=300)

snippet_ex1 = """
import Mathlib.CategoryTheory.Limits.Shapes.Equalizers

#check @CategoryTheory.Limits.HasEqualizers
"""

resultat_ex1 = None  # TODO etudiant : remplacer par run_lean(snippet_ex1, timeout_s=300)
print("Exercice 1 a completer")

Exercice 1 a completer


### Exercice 2 : prouver une identite sur Sieve.pullback

**Objectif** : ecrire et prouver un theoreme simple sur `Sieve.pullback`, inspire de `SieveLattice.lean`.

**Indice** : essayez de prouver `Sieve.pullback f ⊤ = ⊤` (variante de Calibration P2) ou une autre identite du pullback. Pattern standard : `ext Z g` puis `simp [Sieve.pullback]`.

**Etapes** :
1. Choisir une identite a prouver.
2. Ecrire l'enonce dans un snippet Lean.
3. Trouver la preuve.

In [17]:
# Exercice 2 : prouver une identite sur Sieve.pullback
# TODO etudiant : ecrire un theoreme sur Sieve.pullback et le prouver
# Indice : utilisez le pattern ext Z g / simp [Sieve.pullback] des modules SieveLattice/Calibration
# Etape 1 : choisir une identite (ex: pullback along identity, pullback composition, etc.)
# Etape 2 : ecrire le snippet avec l'enonce et la preuve
# Etape 3 : executer avec run_lean()

snippet_ex2 = """
import Mathlib.CategoryTheory.Sites.Grothendieck

open CategoryTheory

theorem pullback_top_variant {C : Type*} [Category C] {X Y : C} (f : Y ⟶ X) :
    (Sieve.pullback f ⊤ : Sieve Y) = ⊤ := by
  -- TODO etudiant : completer la preuve
  -- Indice : essayez ext Z g puis simp [Sieve.pullback]
  sorry
"""

resultat_ex2 = None  # TODO etudiant : remplacer par run_lean(snippet_ex2, timeout_s=300)
print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 : ajouter une micro-preuve a Calibration

**Objectif** : ecrire une nouvelle micro-preuve dans le style de `Calibration.lean`, en utilisant une tactique differente de P1-P4.

**Indice** : essayez de prouver que `GrothendieckTopology.trivial C ≤ J` pour toute topologie `J`. Pattern : `rw [GrothendieckTopology.trivial_eq_bot]` puis `exact bot_le`.

**Etapes** :
1. Choisir un enonce simple sur les topologies.
2. Ecrire le theoreme et la preuve dans un snippet.
3. Executer avec `run_lean()`.

In [18]:
# Exercice 3 : ajouter une micro-preuve a Calibration
# TODO etudiant : ecrire un theoreme et sa preuve sur les cribles/topologies
# Indice : essayez un enonce utilisant native_decide ou rfl
#   Exemple : prouver que le pullback est idempotent pour l'identite
#   ou que le treillis des topologies est ordonne (bot <= J <= top)
# Etape 1 : choisir un enonce
# Etape 2 : ecrire le snippet avec la preuve
# Etape 3 : executer avec run_lean()

snippet_ex3 = """
import Mathlib.CategoryTheory.Sites.Grothendieck

open CategoryTheory

-- P5 candidate : le treillis des topologies est ordonne
theorem bot_le_topology {C : Type*} [Category C] (J : GrothendieckTopology C) :
    GrothendieckTopology.trivial C ≤ J := by
  -- TODO etudiant : completer la preuve
  -- Indice : rw [GrothendieckTopology.trivial_eq_bot]
  sorry
"""

resultat_ex3 = None  # TODO etudiant : remplacer par run_lean(snippet_ex3, timeout_s=300)
print("Exercice 3 a completer")

Exercice 3 a completer


## 10. Conclusion

Ce notebook a explore les 6 modules pedagogiques du projet `grothendieck_lean/` (parmi les 23 modules au total), en mettant l'accent sur la **lecture directe des sources** et l'**analyse des preuves**.

### Recapitulatif

| Module | Lignes | Contenu | Theoremes cles |
|--------|--------|---------|---------------|
| `CategoryAndSites` | 106 | Cribles, topologies, 3 axiomes | `top_covers`, `pullback_cover`, `transitivity` |
| `SchemesTour` | 79 | Scheme, Spec, Gamma | Foncteurs d'oubli, homeomorphisme d'isos |
| `ZariskiSite` | 84 | Pretopologie -> topologie | `zariski_topology_eq`, sous-canonique |
| `MathlibMap` | 90 | Index vivant #check | 16 verifications structurelles |
| `Calibration` | 80 | 4 micro-preuves P1-P4 | `trivial_le_discrete`, `pullback_top`, `zariski_eq_pretopology`, `isSheaf_trivial` |
| `SieveLattice` | 88 | Pullback identities | `pullback_id`, `pullback_pullback`, `pullback_bot`, `pullback_monotone` |

### Points cles a retenir

1. **Le langage de Grothendieck est naturel en Lean** : les definitions de Mathlib epousent celles de SGA 4.
2. **Les preuves sont courtes** : les micro-preuves P1-P4 font 1-3 lignes chacune, mais chacune illustre un pattern different.
3. **Le pullback est central** : 5 des 8 theoremes du projet impliquent `Sieve.pullback`.
4. **Le treillis est complet** : `Sieve X` et `GrothendieckTopology C` sont des treillis complets.
5. **Le projet est sorry = 0** : toutes les preuves sont closes sur les 23 modules.

> **Etendue du projet** : les 17 modules avances supplementaires (Parts 7-23) couvrent les operations sur cribles, generateurs de coverage, proprietes canoniques, topologie dense, faisceautisation et son exactitude a gauche, points d'un site, sous-canonicite, hom-faisceaux, faisceau constant, familles conservatives, cohomologie des faisceaux par Ext, carres de Mayer-Vietoris, suite exacte longue de Mayer-Vietoris et cohomologie de Cech. Ils sont verifies par la cellule d'inventaire ci-dessus (23 modules, 0 sorry).

### Pour aller plus loin

- [Lean-15 (Hommage Grothendieck)](Lean-15-Grothendieck-Tribute.ipynb) : le contexte biographique et mathematique.
- [Lean-16b (Conway Tribute)](Lean-16b-Conway-Game-of-Life-Lean.ipynb) : un autre hommage, meme pattern Python + Lean.
- [Lean-6 (Mathlib Essentials)](Lean-6-Mathlib-Essentials.ipynb) : tour des structures Mathlib.
- [grothendieck_lean/](grothendieck_lean/) : les sources Lean completes.

### References

1. A. Grothendieck, *Elements de geometrie algebrique* (EGA), 1960-1967.
2. A. Grothendieck et al., *Seminaire de geometrie algebrique du Bois-Marie* (SGA 1-7), 1962-1969.
3. The Stacks Project, https://stacks.math.columbia.edu/
4. R. Vakil, *The Rising Sea*, https://math.stanford.edu/~vakil/216blog/
5. Mathlib 4 documentation, https://leanprover-community.github.io/mathlib4_docs/

---

**Navigation** : [<< Lean-15 Grothendieck Tribute](Lean-15-Grothendieck-Tribute.ipynb) | [Lean-16b Conway Tribute >>](Lean-16b-Conway-Game-of-Life-Lean.ipynb) | [Index](README.md)